# 06-6. DNS와 주소 해석 예제

## Goal

- 호스트 이름과 해석된 IP 주소를 구분합니다.
- 모든 해석 결과가 허용 범위인지 검사합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

`localhost`만 조회하고 외부 DNS에는 질의하지 않습니다.


## Steps

### 주소 해석 결과 정규화

중복 주소를 제거하고 IPv4·IPv6 루프백 여부를 검사합니다.


In [1]:
import ipaddress
import socket


def resolved_addresses(host: str, port: int) -> tuple[str, ...]:
    records = socket.getaddrinfo(host, port, type=socket.SOCK_STREAM)
    addresses = {ipaddress.ip_address(record[4][0]).compressed for record in records}
    return tuple(sorted(addresses, key=lambda value: (ipaddress.ip_address(value).version, value)))


addresses = resolved_addresses("localhost", 80)
print("localhost 해석 결과:", addresses)
print("모두 루프백:", all(ipaddress.ip_address(value).is_loopback for value in addresses))


localhost 해석 결과: ('127.0.0.1', '::1')
모두 루프백: True


## Checks

빈 결과와 외부 주소를 허용하지 않는 정책을 함수로 분리합니다.


In [2]:
def require_loopback(addresses):
    if not addresses:
        raise ValueError("주소 해석 결과가 없습니다")
    parsed = [ipaddress.ip_address(value) for value in addresses]
    if not all(address.is_loopback for address in parsed):
        raise ValueError("루프백이 아닌 주소가 포함되어 있습니다")
    return tuple(address.compressed for address in parsed)


assert require_loopback(addresses)
try:
    require_loopback(["127.0.0.1", "192.0.2.10"])
except ValueError:
    print("외부 주소 혼합 거부 확인")


외부 주소 혼합 거부 확인


## Next Steps

실제 서비스에서는 연결 직전에도 해석 결과와 허용 범위를 다시 확인합니다.
